Para abrir o notebook no Google Colab, altere o domínio `github.com` para `githubtocolab.com`

<div class="alert alert-block alert-danger">
Para praticar programação, é importante que você erre, leia as mensagens de erro e tente corrigí-los.
    
Dessa forma, no Google Colab, é importante que você DESATIVE OS RECURSOS DE AUTOCOMPLETAR:

- Menu Ferramentas -> Configurações
- Na janela que é aberta:
  - Seção Editor -> Desativar "Mostrar sugestões de preenchimento de código com base no contexto"
  - Seção Assistência de IA -> Desabilitar itens

Na versão em inglês:

- Menu Tools -> Settings
- Na janela que é aberta:
  - Seção Editor -> Desativar "Show context-powered code completions"
  - Seção AI Assistance -> Desabilitar itens
</div>

# Introdução ao Python para processamento numérico e análise de dados

# Tópicos sobre o *framework* PyTorch

No Google Colab, é necessário habilitar o suporte à GPU acessando "Change Runtime Type" e selecionando uma opção com GPU como, por exemplo, a "T4 GPU".

O [PyTorch](https://pytorch.org/) é um dos *frameworks* mais utilizados para o treinamento de modelos de aprendizado de máquina. A seguir, são listados os seus principais componentes e é mostrado um exemplo evoluindo de um algoritmo adaptativo implementado em NumPy, incorporando componentes do PyTorch um a um, até chegar em uma implementação clássica do algoritmo usando PyTorch.

Antes disso, vamos carregar as bibliotecas necessárias:

In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from scipy import signal
from torch import nn

## 1. Tensores

- Objetos do tipo `torch.Tensor`, semelhantes à *arrays* do NumPy, mas com algumas funções adicionais:
  - Podem ser alocados facilmente na GPU;
  - Possibilidade de cálculo automático de gradientes.
- Ref.: [https://pytorch.org/docs/stable/tensors.html](https://pytorch.org/docs/stable/tensors.html)

- Podem ser criados a partir de listas do Python:

In [2]:
torch.tensor([2, 3, 4])

tensor([2, 3, 4])

- Atributos de *shape* e *rank* semelhantes aos *arrays* do NumPy:

In [3]:
x = torch.tensor([2, 3, 4])
x.shape

torch.Size([3])

In [4]:
x = torch.tensor([[2, 3, 4]])
x.shape

torch.Size([1, 3])

- Tipo padrão para representação de ponto flutuante é o `float32`:

In [5]:
x = torch.tensor([[2., 3., 4.]])
x.dtype

torch.float32

- Funções auxiliares para criação de tensores com zeros, uns, aleatórios e matrizes identidade:

In [6]:
torch.zeros((2, 3))

tensor([[0., 0., 0.],
        [0., 0., 0.]])

In [7]:
torch.ones((3, 2))

tensor([[1., 1.],
        [1., 1.],
        [1., 1.]])

In [8]:
torch.rand(2, 2)

tensor([[0.8559, 0.5938],
        [0.3894, 0.3287]])

In [9]:
torch.randn(3, 3)

tensor([[ 1.5333,  0.4722, -2.6308],
        [-0.7967, -1.1563, -1.3638],
        [ 1.2577,  0.3318,  0.5305]])

In [10]:
torch.eye(3)

tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])

- Funções auxiliares para criação de sequências:

In [11]:
torch.linspace(0, 1, 10)

tensor([0.0000, 0.1111, 0.2222, 0.3333, 0.4444, 0.5556, 0.6667, 0.7778, 0.8889,
        1.0000])

In [12]:
torch.arange(10)

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

- `reshape`e `view`: são semelhantes, mas
  - `view` usa os mesmos dados do tensor original. Não funciona para o caso de dados não contíguos;
  - `reshape` tenta fazer o mesmo que `view`, mas no caso de dados não contíguos, retorna um tensor com uma cópia dos dados originais.
- A sugestão é usar sempre `view`;
- Ref.: [https://stackoverflow.com/questions/49643225/whats-the-difference-between-reshape-and-view-in-pytorch](https://stackoverflow.com/questions/49643225/whats-the-difference-between-reshape-and-view-in-pytorch)

In [13]:
x = torch.rand(4,3)
x

tensor([[0.0293, 0.6819, 0.2354],
        [0.6511, 0.7750, 0.4751],
        [0.9464, 0.1122, 0.6902],
        [0.3112, 0.3120, 0.2114]])

In [14]:
x.view(12, -1)

tensor([[0.0293],
        [0.6819],
        [0.2354],
        [0.6511],
        [0.7750],
        [0.4751],
        [0.9464],
        [0.1122],
        [0.6902],
        [0.3112],
        [0.3120],
        [0.2114]])

In [15]:
x.reshape(12, -1)

tensor([[0.0293],
        [0.6819],
        [0.2354],
        [0.6511],
        [0.7750],
        [0.4751],
        [0.9464],
        [0.1122],
        [0.6902],
        [0.3112],
        [0.3120],
        [0.2114]])

## 2. Operações

- Operações aritméticas e matriciais semelhantes às do NumPy;
- PyTorch disponibiliza diversas operações ponto a ponto como `torch.abs()` e `torch.cos()` e diversas operações de redução como `torch.sum()` e `torch.mean()`;
- É importante utilizar as operações do PyTorch para processar os tensores, para que seja possível calcular o gradiente automaticamente com o autograd;
- Além disso, há diversas operações de comparação, espectrais e outras.
- Referência: [https://pytorch.org/docs/stable/torch.html#math-operations](https://pytorch.org/docs/stable/torch.html#math-operations)

## 3. Alocação em CPU e em GPU

- Atributo `is_cuda` permite ver se o tensor está alocado na GPU:

In [16]:
x = torch.Tensor([1, 2, 3])

In [17]:
x.is_cuda

False

- Para alocar na GPU, é necessário criar um objeto device e usar o método `.to()`:

In [18]:
device = torch.device("cuda")
x = x.to(device)

In [19]:
x.is_cuda

True

- É possível ver se há GPU disponível com o método `torch.cuda.is_available()`:

In [20]:
torch.cuda.is_available() 

True

- É usual usar a seguinte estrutura para alocação automática de tensores na GPU, quando disponível:

In [21]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
x = x.to(device)

- Para trazer realocar um tensor de volta à CPU, pode-se usar `.cpu()`:

In [22]:
x.is_cuda

True

In [23]:
x = x.cpu()

In [24]:
x.is_cuda

False

- O método `.to()` também é usado para fazer *casting* de tensores:

In [25]:
x.dtype

torch.float32

In [26]:
y = x.to(torch.float64)
y.dtype

torch.float64

## 4. Conversão de dados para NumPy e vice-versa

- Um tensor do PyTorch pode ser convertido para um *array* do NumPy usando o método `.numpy()`:

In [27]:
x = torch.randn(5,5)
x

tensor([[ 0.6241,  0.2994, -1.7575,  1.4482, -0.3789],
        [ 0.9354,  0.4998,  0.0549,  0.8552,  0.9118],
        [ 0.4448, -1.7049, -0.3977, -0.7377,  0.2000],
        [-0.5846,  0.5489,  2.0510,  0.1561, -0.9279],
        [ 0.6917,  0.9258, -0.4977,  0.0616, -0.5681]])

In [28]:
x.dtype

torch.float32

In [29]:
x_np = x.numpy()
x_np

array([[ 0.6240946 ,  0.2994123 , -1.7574902 ,  1.4481531 , -0.3788658 ],
       [ 0.935421  ,  0.49981344,  0.05494728,  0.855241  ,  0.91175085],
       [ 0.44476125, -1.7049137 , -0.39767736, -0.7376809 ,  0.19995019],
       [-0.5846313 ,  0.5489112 ,  2.0509903 ,  0.15606847, -0.92791045],
       [ 0.69170547,  0.9258369 , -0.49769092,  0.0615948 , -0.5680976 ]],
      dtype=float32)

In [30]:
x_np.dtype

dtype('float32')

- `torch.Tensor` pode criar um tensor PyTorch a partir de um *array* do NumPy, mas é necessário atenção à precisão numérica:

In [31]:
x_np = np.random.randn(5,5)
x_np

array([[ 1.63969104, -0.74702607,  0.80403545, -0.59040857,  0.91201127],
       [-0.03437457,  0.33726328, -0.33778283,  0.44526212, -0.918775  ],
       [ 0.99036412, -1.4990199 ,  0.61361759, -0.72722151,  1.12309093],
       [ 0.35252627,  0.92200162, -1.17453363,  1.73436042, -0.02326564],
       [-0.33380103, -1.36188074, -1.71488459, -0.39055168,  0.79606723]])

In [32]:
x_np.dtype

dtype('float64')

In [33]:
x = torch.Tensor(x_np,)
x

tensor([[ 1.6397, -0.7470,  0.8040, -0.5904,  0.9120],
        [-0.0344,  0.3373, -0.3378,  0.4453, -0.9188],
        [ 0.9904, -1.4990,  0.6136, -0.7272,  1.1231],
        [ 0.3525,  0.9220, -1.1745,  1.7344, -0.0233],
        [-0.3338, -1.3619, -1.7149, -0.3906,  0.7961]])

In [34]:
x.dtype

torch.float32

- A função `torch.from_numpy` preserva o tipo do *array* NumPy:

In [35]:
x2 = torch.from_numpy(x_np)
x2.dtype

torch.float64

## 5. Autograd

- Tensores com o atributo `requires_grad=True` têm o gradiente calculado automaticamente;
- Só vetores do tipo `float` ou `complex` podem usar `requires_grad=True`.

In [36]:
x0 = torch.tensor([1., 2., 3.], requires_grad=True)
x0

tensor([1., 2., 3.], requires_grad=True)

In [37]:
x1 = torch.tensor([4., 5., 6.], requires_grad=True)
x1

tensor([4., 5., 6.], requires_grad=True)

- Tensores criados a partir de outros com `requires_grad=True` também têm `requires_grad=True`:

In [38]:
f = torch.sum(x0**2 + x1)

In [39]:
f.requires_grad

True

- Tensores criados pelo usuário são *leaf nodes* no grafo, identificados pelo atributo `is_leaf`:

In [40]:
x0.is_leaf

True

In [41]:
x1.is_leaf

True

In [42]:
f.is_leaf

False

- Gradiente de `f` em relação `x0` e `x1`:

$$
\frac{\partial f}{\partial \mathbf{x}} = 
\left[
\begin{matrix}
\frac{\partial f}{\partial x_0}\\
\frac{\partial f}{\partial x_1}\\
\end{matrix}
\right] = 
\left[
\begin{matrix}
2x_0\\
1\\
\end{matrix}
\right]
$$

- Os gradientes são armazenados no atributo `grad`, inicialmente igual a `None`:

In [43]:
x0.grad is None

True

In [44]:
x1.grad is None

True

- Para que os gradientes sejam calculados, é necessário executar o método `.backward()` do nó em relação ao qual se deseja calculá-los:

In [45]:
f.backward()

In [46]:
x0.grad

tensor([2., 4., 6.])

In [47]:
x0

tensor([1., 2., 3.], requires_grad=True)

In [48]:
x1.grad

tensor([1., 1., 1.])

In [49]:
x1

tensor([4., 5., 6.], requires_grad=True)